# GPR Inversion Thesis - Step 3
## FNO Training + U-Net Baseline Comparison
**Author:** Tonmoy Neog — MSc (Tech) Geophysics, BHU

---
### What this notebook does:
1. Loads the 1000-sample dataset from Google Drive
2. Splits into 800 train / 100 val / 100 test
3. Trains a Fourier Neural Operator (FNO)
4. Trains a U-Net baseline
5. Compares both models quantitatively and visually
6. Saves all results and figures to Google Drive
---

## CELL 1 - Mount Drive and Install Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'numpy', 'matplotlib', 'scikit-learn', 'scipy'], capture_output=True)

import torch
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.ndimage import zoom

print(f'PyTorch version: {torch.__version__}')
print(f'GPU available:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name:        {torch.cuda.get_device_name(0)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

THESIS_DIR  = '/content/drive/MyDrive/GPR_Thesis'
DATASET_DIR = f'{THESIS_DIR}/dataset'
RESULTS_DIR = f'{THESIS_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Load dataset ──────────────────────────────────────────────────────────────
print('\nLoading dataset...')
bscans   = np.load(f'{DATASET_DIR}/bscans_all.npy')    # (1000, 512, 100)
eps_maps = np.load(f'{DATASET_DIR}/eps_maps_all.npy')  # (1000, 200, 100)
print(f'B-scans:           {bscans.shape}')
print(f'Permittivity maps: {eps_maps.shape}')

# ── Normalize eps maps to [0, 1] globally ─────────────────────────────────────
eps_min  = eps_maps.min()
eps_max  = eps_maps.max()
eps_norm = (eps_maps - eps_min) / (eps_max - eps_min)
print(f'\nEps range: [{eps_min:.2f}, {eps_max:.2f}] -> normalized to [0, 1]')
np.save(f'{RESULTS_DIR}/eps_min_max.npy', np.array([eps_min, eps_max]))

# ── Resize to 128x128 ─────────────────────────────────────────────────────────
print('\nResizing to 128x128...')
bscans_r = np.array([zoom(b, (128/512, 128/100), order=1) for b in bscans])
eps_r    = np.array([zoom(e, (128/200, 128/100), order=1) for e in eps_norm])
print(f'Resized B-scans:  {bscans_r.shape}')
print(f'Resized eps maps: {eps_r.shape}')

# ── Per-sample normalization of B-scans to [-1, 1] ────────────────────────────
# This is CRITICAL for FNO convergence
# Each B-scan has very different amplitude ranges
# Without this FNO cannot learn meaningful Fourier modes
print('\nApplying per-sample normalization to B-scans...')
bscans_norm = np.zeros_like(bscans_r, dtype=np.float32)
for i in range(len(bscans_r)):
    mx = np.abs(bscans_r[i]).max()
    if mx > 0:
        bscans_norm[i] = bscans_r[i] / mx
    else:
        bscans_norm[i] = bscans_r[i]
print(f'B-scan range after normalization: [{bscans_norm.min():.3f}, {bscans_norm.max():.3f}]')
print(f'Eps map range:                    [{eps_r.min():.3f}, {eps_r.max():.3f}]')

# ── Convert to tensors ────────────────────────────────────────────────────────
X = torch.FloatTensor(bscans_norm).unsqueeze(1)  # (1000, 1, 128, 128)
Y = torch.FloatTensor(eps_r).unsqueeze(1)         # (1000, 1, 128, 128)
print(f'\nX tensor: {X.shape}')
print(f'Y tensor: {Y.shape}')

# ── Split 800 / 100 / 100 ────────────────────────────────────────────────────
class GPRDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

dataset = GPRDataset(X, Y)
train_ds, val_ds, test_ds = random_split(
    dataset, [800, 100, 100],
    generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print(f'\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print('Dataset ready!')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch version: 2.10.0+cu128
GPU available:   True
GPU name:        Tesla T4

Loading dataset...
B-scans:           (1000, 512, 100)
Permittivity maps: (1000, 200, 100)

Eps range: [1.00, 81.00] -> normalized to [0, 1]

Resizing to 128x128...
Resized B-scans:  (1000, 128, 128)
Resized eps maps: (1000, 128, 128)

Applying per-sample normalization to B-scans...
B-scan range after normalization: [-1.000, 1.000]
Eps map range:                    [0.000, 1.000]

X tensor: torch.Size([1000, 1, 128, 128])
Y tensor: torch.Size([1000, 1, 128, 128])

Train: 800 | Val: 100 | Test: 100
Dataset ready!


## CELL 2 - Load Dataset and Split

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── FNO ───────────────────────────────────────────────────────────────────────
class SpectralConv2d(nn.Module):
    """2D Fourier layer -- core of the FNO."""
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale  = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum('bixy,ioxy->boxy', input, weights)

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft   = torch.fft.rfft2(x)
        out_ft = torch.zeros(B, self.out_channels, H, W//2+1,
                             dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        return torch.fft.irfft2(out_ft, s=(H, W))


class FNO2d(nn.Module):
    """Fourier Neural Operator for GPR inverse problem."""
    def __init__(self, modes1=16, modes2=16, width=32, n_layers=4):
        super().__init__()
        self.fc0   = nn.Linear(1, width)
        self.convs = nn.ModuleList([
            SpectralConv2d(width, width, modes1, modes2) for _ in range(n_layers)
        ])
        self.ws = nn.ModuleList([
            nn.Conv2d(width, width, 1) for _ in range(n_layers)
        ])
        self.fc1 = nn.Linear(width, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)        # (B, H, W, 1)
        x = self.fc0(x)                   # (B, H, W, width)
        x = x.permute(0, 3, 1, 2)        # (B, width, H, W)
        for conv, w in zip(self.convs, self.ws):
            x = F.gelu(conv(x) + w(x))
        x = x.permute(0, 2, 3, 1)        # (B, H, W, width)
        x = F.gelu(self.fc1(x))          # (B, H, W, 128)
        x = self.fc2(x)                   # (B, H, W, 1)
        return x.permute(0, 3, 1, 2)     # (B, 1, H, W)


# ── U-Net ─────────────────────────────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.downs      = nn.ModuleList()
        self.ups        = nn.ModuleList()
        self.pool       = nn.MaxPool2d(2, 2)
        ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(ch, f))
            ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.ups.append(DoubleConv(f*2, f))
        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i//2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i+1](x)
        return self.final(x)


# ── Instantiate both models ───────────────────────────────────────────────────
fno_model  = FNO2d(modes1=16, modes2=16, width=32, n_layers=4).to(device)
unet_model = UNet().to(device)

fno_params  = sum(p.numel() for p in fno_model.parameters())
unet_params = sum(p.numel() for p in unet_model.parameters())
print(f'FNO  parameters: {fno_params:,}')
print(f'UNet parameters: {unet_params:,}')

# Quick sanity check
test_input = torch.randn(2, 1, 128, 128).to(device)
with torch.no_grad():
    fno_out  = fno_model(test_input)
    unet_out = unet_model(test_input)
print(f'FNO  output shape: {fno_out.shape}   <- correct: (2, 1, 128, 128)')
print(f'UNet output shape: {unet_out.shape}   <- correct: (2, 1, 128, 128)')
print('Both models verified and ready!')


FNO  parameters: 2,105,793
UNet parameters: 7,765,409
FNO  output shape: torch.Size([2, 1, 128, 128])   <- correct: (2, 1, 128, 128)
UNet output shape: torch.Size([2, 1, 128, 128])   <- correct: (2, 1, 128, 128)
Both models verified and ready!


## CELL 3 - Define FNO Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpectralConv2d(nn.Module):
    """2D Fourier layer -- core of the FNO."""
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale  = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum('bixy,ioxy->boxy', input, weights)

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = torch.fft.rfft2(x)
        out_ft = torch.zeros(B, self.out_channels, H, W//2+1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        return torch.fft.irfft2(out_ft, s=(H, W))


class FNO2d(nn.Module):
    """Fourier Neural Operator for GPR inverse problem."""
    def __init__(self, modes1=12, modes2=12, width=32, n_layers=4):
        super().__init__()
        self.modes1  = modes1
        self.modes2  = modes2
        self.width   = width
        self.n_layers = n_layers

        # Lifting layer: map input channels to width
        self.fc0 = nn.Linear(1, width)

        # Fourier layers
        self.convs = nn.ModuleList([
            SpectralConv2d(width, width, modes1, modes2)
            for _ in range(n_layers)
        ])
        self.ws = nn.ModuleList([
            nn.Conv2d(width, width, 1)
            for _ in range(n_layers)
        ])

        # Projection layers: map width to output
        self.fc1 = nn.Linear(width, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        # x: (B, 1, H, W)
        x = x.permute(0, 2, 3, 1)   # (B, H, W, 1)
        x = self.fc0(x)              # (B, H, W, width)
        x = x.permute(0, 3, 1, 2)   # (B, width, H, W)

        for i in range(self.n_layers):
            x1 = self.convs[i](x)
            x2 = self.ws[i](x)
            x  = F.gelu(x1 + x2)

        x = x.permute(0, 2, 3, 1)   # (B, H, W, width)
        x = F.gelu(self.fc1(x))     # (B, H, W, 128)
        x = self.fc2(x)             # (B, H, W, 1)
        x = x.permute(0, 3, 1, 2)   # (B, 1, H, W)
        return x


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
fno_model = FNO2d(modes1=12, modes2=12, width=32, n_layers=4).to(device)
total_params = sum(p.numel() for p in fno_model.parameters())
print(f'FNO model created on: {device}')
print(f'Total parameters:     {total_params:,}')
print(f'Modes: 12x12 | Width: 32 | Layers: 4')


FNO model created on: cuda
Total parameters:     1,188,289
Modes: 12x12 | Width: 32 | Layers: 4


## CELL 4 - Define U-Net Baseline Model

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.downs    = nn.ModuleList()
        self.ups      = nn.ModuleList()
        self.pool     = nn.MaxPool2d(2, 2)

        # Encoder
        ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(ch, f))
            ch = f

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        # Decoder
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.ups.append(DoubleConv(f*2, f))

        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i//2]
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i+1](x)
        return self.final(x)

unet_model = UNet().to(device)
unet_params = sum(p.numel() for p in unet_model.parameters())
print(f'U-Net model created on: {device}')
print(f'Total parameters:       {unet_params:,}')


U-Net model created on: cuda
Total parameters:       7,765,409


## CELL 5 - Training Function

In [ ]:
## CELL 5 - Training Function

import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

def rel_l2_loss(pred, target):
    return torch.mean(
        torch.norm(pred.view(pred.size(0), -1) - target.view(target.size(0), -1), dim=1) /
        torch.norm(target.view(target.size(0), -1), dim=1)
    )

def train_model(model, train_loader, val_loader, model_name='model', n_epochs=300, lr=1e-4):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
    criterion = rel_l2_loss

    train_losses, val_losses = [], []
    best_val = float('inf')

    for epoch in range(n_epochs):
        model.train()
        batch_train = []
        for X_batch, Y_batch in train_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), Y_batch)
            loss.backward()
            optimizer.step()
            batch_train.append(loss.item())

        model.eval()
        batch_val = []
        with torch.no_grad():
            for X_batch, Y_batch in val_loader:
                X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
                batch_val.append(criterion(model(X_batch), Y_batch).item())

        epoch_train = np.mean(batch_train)
        epoch_val   = np.mean(batch_val)
        train_losses.append(epoch_train)
        val_losses.append(epoch_val)
        scheduler.step(epoch_val)

        if epoch_val < best_val:
            best_val = epoch_val
            torch.save(model.state_dict(), f'{RESULTS_DIR}/{model_name}_best.pt')

        if (epoch + 1) % 25 == 0:
            print(f'[{model_name}] Epoch {epoch+1}/{n_epochs} | '
                  f'Train: {epoch_train:.6f} | Val: {epoch_val:.6f} | Best: {best_val:.6f}')

    return train_losses, val_losses, best_val

print('train_model() defined with relative L2 loss — ready.')

train_model() defined with relative L2 loss — ready.


## CELL 6 - Train FNO (takes ~45-60 min on T4)

In [ ]:
## CELL 6 - Train FNO (takes ~45-60 min on T4)

fno_train_losses, fno_val_losses, fno_best_val = train_model(
    fno_model, train_loader, val_loader,
    model_name='fno',
    n_epochs=300,
    lr=1e-4
)
np.save(f'{RESULTS_DIR}/fno_train_losses.npy', np.array(fno_train_losses))
np.save(f'{RESULTS_DIR}/fno_val_losses.npy',   np.array(fno_val_losses))
print('FNO losses saved to Drive.')

[fno] Epoch 25/300 | Train: 0.452150 | Val: 0.434161 | Best: 0.432848
[fno] Epoch 50/300 | Train: 0.365466 | Val: 0.361654 | Best: 0.359271
[fno] Epoch 75/300 | Train: 0.330475 | Val: 0.333398 | Best: 0.333398
[fno] Epoch 100/300 | Train: 0.310447 | Val: 0.321347 | Best: 0.320794
[fno] Epoch 125/300 | Train: 0.295239 | Val: 0.317607 | Best: 0.314306
[fno] Epoch 150/300 | Train: 0.280383 | Val: 0.312216 | Best: 0.309794
[fno] Epoch 175/300 | Train: 0.267992 | Val: 0.309847 | Best: 0.306810
[fno] Epoch 200/300 | Train: 0.255239 | Val: 0.306122 | Best: 0.302488
[fno] Epoch 225/300 | Train: 0.244639 | Val: 0.301077 | Best: 0.300146
[fno] Epoch 250/300 | Train: 0.233049 | Val: 0.298229 | Best: 0.296431
[fno] Epoch 275/300 | Train: 0.221246 | Val: 0.293918 | Best: 0.293783
[fno] Epoch 300/300 | Train: 0.210468 | Val: 0.290048 | Best: 0.289377
FNO losses saved to Drive.


## CELL 7 - Train U-Net (takes ~45-60 min on T4)

In [ ]:
## CELL 7 - Train U-Net (takes ~45-60 min on T4)

unet_train_losses, unet_val_losses, unet_best_val = train_model(
    unet_model, train_loader, val_loader,
    model_name='unet',
    n_epochs=300,
    lr=1e-3
)
np.save(f'{RESULTS_DIR}/unet_train_losses.npy', np.array(unet_train_losses))
np.save(f'{RESULTS_DIR}/unet_val_losses.npy',   np.array(unet_val_losses))
print('U-Net losses saved to Drive.')

NameError: name 'train_model' is not defined

## CELL 8 - Evaluate Both Models on Test Set

In [ ]:
import numpy as np
import torch
from sklearn.metrics import mean_absolute_error

def evaluate_model(model, model_name, test_loader):
    # Load best saved weights
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/{model_name}_best.pt', map_location=device))
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_batch, Y_batch in test_loader:
            pred = model(X_batch.to(device)).cpu().numpy()
            all_preds.append(pred)
            all_targets.append(Y_batch.numpy())

    preds   = np.concatenate(all_preds,   axis=0)  # (100, 1, 64, 64)
    targets = np.concatenate(all_targets, axis=0)

    # Metrics
    mse  = np.mean((preds - targets)**2)
    mae  = np.mean(np.abs(preds - targets))
    rmse = np.sqrt(mse)

    # Relative L2 error (standard FNO metric)
    rel_l2 = np.mean(
        np.linalg.norm(preds.reshape(len(preds), -1) - targets.reshape(len(targets), -1), axis=1) /
        np.linalg.norm(targets.reshape(len(targets), -1), axis=1)
    )

    print(f'{model_name.upper()} Test Results:')
    print(f'  MSE:          {mse:.6f}')
    print(f'  RMSE:         {rmse:.6f}')
    print(f'  MAE:          {mae:.6f}')
    print(f'  Relative L2:  {rel_l2:.4f} ({rel_l2*100:.2f}%)')

    return preds, targets, {'mse': mse, 'rmse': rmse, 'mae': mae, 'rel_l2': rel_l2}

print('Evaluating FNO...')
fno_preds, targets, fno_metrics = evaluate_model(fno_model, 'fno', test_loader)
print()
print('Evaluating U-Net...')
unet_preds, _, unet_metrics = evaluate_model(unet_model, 'unet', test_loader)
print()
print('Comparison:')
print(f'  Relative L2 -- FNO: {fno_metrics["rel_l2"]*100:.2f}%  |  U-Net: {unet_metrics["rel_l2"]*100:.2f}%')
winner = 'FNO' if fno_metrics['rel_l2'] < unet_metrics['rel_l2'] else 'U-Net'
print(f'  Winner: {winner}')

# Save metrics
import json
with open(f'{RESULTS_DIR}/metrics.json', 'w') as f:
    json.dump({'fno': {k: float(v) for k,v in fno_metrics.items()},
               'unet': {k: float(v) for k,v in unet_metrics.items()}}, f, indent=2)
print('Metrics saved.')


## CELL 9 - Plot Loss Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training and Validation Loss Curves', fontsize=13, fontweight='bold')

epochs = range(1, len(fno_train_losses) + 1)

axes[0].plot(epochs, fno_train_losses, 'b-',  linewidth=1.5, label='Train Loss')
axes[0].plot(epochs, fno_val_losses,   'r--', linewidth=1.5, label='Val Loss')
axes[0].set_title('FNO', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

axes[1].plot(epochs, unet_train_losses, 'b-',  linewidth=1.5, label='Train Loss')
axes[1].plot(epochs, unet_val_losses,   'r--', linewidth=1.5, label='Val Loss')
axes[1].set_title('U-Net', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print('Loss curves saved.')


## CELL 10 - Visual Comparison: Prediction vs Ground Truth

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_examples = 4
fig, axes = plt.subplots(n_examples, 4, figsize=(16, 4*n_examples))
fig.suptitle('FNO vs U-Net: Predicted Permittivity Maps vs Ground Truth\n'
             '(Test set examples)', fontsize=13, fontweight='bold')

col_titles = ['Input B-scan', 'Ground Truth', 'FNO Prediction', 'U-Net Prediction']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11, fontweight='bold')

# Get test samples
test_X, test_Y = [], []
for X_batch, Y_batch in test_loader:
    test_X.append(X_batch)
    test_Y.append(Y_batch)
    if len(test_X) * 16 >= n_examples:
        break
test_X = torch.cat(test_X)[:n_examples]
test_Y = torch.cat(test_Y)[:n_examples]

fno_model.eval()
unet_model.eval()
with torch.no_grad():
    fno_out  = fno_model(test_X.to(device)).cpu().numpy()
    unet_out = unet_model(test_X.to(device)).cpu().numpy()

for i in range(n_examples):
    bscan_img = test_X[i, 0].numpy()
    gt_img    = test_Y[i, 0].numpy()
    fno_img   = fno_out[i, 0]
    unet_img  = unet_out[i, 0]

    clip = 0.05 * np.abs(bscan_img).max()
    axes[i, 0].imshow(np.clip(bscan_img, -clip, clip), cmap='RdBu', aspect='auto')
    axes[i, 0].set_ylabel(f'Sample {i+1}', fontsize=10)

    im1 = axes[i, 1].imshow(gt_img,   cmap='viridis', aspect='auto', vmin=0, vmax=1)
    im2 = axes[i, 2].imshow(fno_img,  cmap='viridis', aspect='auto', vmin=0, vmax=1)
    im3 = axes[i, 3].imshow(unet_img, cmap='viridis', aspect='auto', vmin=0, vmax=1)

    fno_err  = np.mean((fno_img  - gt_img)**2)
    unet_err = np.mean((unet_img - gt_img)**2)
    axes[i, 2].set_xlabel(f'MSE={fno_err:.5f}',  fontsize=9)
    axes[i, 3].set_xlabel(f'MSE={unet_err:.5f}', fontsize=9)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.colorbar(im1, ax=axes[:, 1], label='Normalised eps_r')
plt.colorbar(im2, ax=axes[:, 2], label='Normalised eps_r')
plt.colorbar(im3, ax=axes[:, 3], label='Normalised eps_r')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/predictions_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Prediction comparison figure saved.')


## CELL 11 - Print Thesis-Ready Results Summary

In [ ]:
import json
with open(f'{RESULTS_DIR}/metrics.json') as f:
    metrics = json.load(f)

fno  = metrics['fno']
unet = metrics['unet']

print('=' * 65)
print('RESULTS SUMMARY - Copy into Chapter 6 of thesis')
print('=' * 65)
print()
print(f'{"Metric":<20} {"FNO":>15} {"U-Net":>15}')
print('-' * 50)
print(f'{"MSE":<20} {fno["mse"]:>15.6f} {unet["mse"]:>15.6f}')
print(f'{"RMSE":<20} {fno["rmse"]:>15.6f} {unet["rmse"]:>15.6f}')
print(f'{"MAE":<20} {fno["mae"]:>15.6f} {unet["mae"]:>15.6f}')
print(f'{"Relative L2 (%)":<20} {fno["rel_l2"]*100:>14.2f}% {unet["rel_l2"]*100:>14.2f}%')
print()
winner = 'FNO' if fno['rel_l2'] < unet['rel_l2'] else 'U-Net'
improvement = abs(fno['rel_l2'] - unet['rel_l2']) / max(fno['rel_l2'], unet['rel_l2']) * 100
print(f'Winner: {winner} (by {improvement:.1f}% on Relative L2 error)')
print()
print('Files saved to Google Drive/GPR_Thesis/results/:')
print('  fno_best.pt               <- trained FNO weights')
print('  unet_best.pt              <- trained U-Net weights')
print('  loss_curves.png           <- Figure 6.X')
print('  predictions_comparison.png <- Figure 6.X')
print('  metrics.json              <- all numbers')
print('=' * 65)
